# 03 - Model Training and Comparison

This notebook compares a RandomForest baseline with LightGBM on both feature sets, then keeps the best result for the assignment step.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

current = Path.cwd().resolve()
candidate_roots = [current, current.parent, current.parent.parent]
ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'smartassign_pipeline.py').exists():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError('Could not locate the repository root.')

sys.path.insert(0, str(ROOT / 'src'))
from smartassign_pipeline import (
    FIGURES_DIR,
    MODELS_DIR,
    RESULTS_DIR,
    ensure_output_directories,
    feature_importance_dataframe,
    fit_and_evaluate_model,
    get_feature_sets,
    load_raw_data,
    save_csv,
    save_json,
    split_train_test,
    summarize_metrics_table,
)

ensure_output_directories()
pd.set_option('display.max_columns', None)

# Train the two requested model families on both feature sets.
df = load_raw_data()
feature_sets = get_feature_sets(df)

results = []
bundles = {}
for feature_set_name, feature_columns in feature_sets.items():
    train_x, test_x, train_y, test_y = split_train_test(df, feature_columns)
    for model_name in ['random_forest', 'lightgbm']:
        bundle, metrics_frame = fit_and_evaluate_model(
            train_x=train_x,
            test_x=test_x,
            train_y=train_y,
            test_y=test_y,
            model_name=model_name,
            feature_set_name=feature_set_name,
        )
        bundles[(feature_set_name, model_name)] = bundle
        results.append(metrics_frame)

metrics = summarize_metrics_table(pd.concat(results, ignore_index=True))
display(metrics)
save_csv(metrics, RESULTS_DIR / 'model_metrics.csv')

best_overall_row = metrics.sort_values(['rmse', 'mae', 'r2'], ascending=[True, True, False]).iloc[0]
best_rf_row = metrics[metrics['model'] == 'random_forest'].sort_values(['rmse', 'mae', 'r2'], ascending=[True, True, False]).iloc[0]
best_bundle = bundles[(best_rf_row['feature_set'], best_rf_row['model'])]
best_bundle.save(MODELS_DIR / 'best_model_bundle.joblib')
save_json(
    {
        'best_overall': best_overall_row.to_dict(),
        'final_model_for_hitl': best_rf_row.to_dict(),
    },
    RESULTS_DIR / 'best_model_metadata.json',
)

best_importances = feature_importance_dataframe(best_bundle, top_n=20)
display(best_importances)
save_csv(best_importances, RESULTS_DIR / 'best_model_feature_importances.csv')

plt.figure(figsize=(12, 6))
plot_data = metrics.copy()
plot_data['label'] = plot_data['feature_set'] + ' / ' + plot_data['model']
plt.bar(plot_data['label'], plot_data['rmse'], color='#5B9BD5')
plt.xticks(rotation=25, ha='right')
plt.ylabel('Test RMSE')
plt.title('Model Comparison Across Feature Sets')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_model_comparison_rmse.png', dpi=150, bbox_inches='tight')
plt.show()

summary_lines = [
    f'Best overall model by RMSE: {best_overall_row["model"]} on the {best_overall_row["feature_set"]} feature set.',
    f'Final model chosen for the HITL workflow: {best_rf_row["model"]} on the {best_rf_row["feature_set"]} feature set.',
    'RandomForest is kept for the final pipeline because it supports per-tree uncertainty estimates for the review stage.',
    'Feature importances are saved for the selected final model so the report can explain which variables drive predictions.',
]
display(Markdown('### Model Training Summary\n' + '\n'.join(f'- {line}' for line in summary_lines)))